<div style="background: #f48fb1; padding: 30px; border-radius: 12px; text-align: center; margin-bottom: 10px;">
  <h1 style="color: white; font-size: 2.4em; margin: 0;">Equalizador Digital de Áudio</h1>
  <h3 style="color: white; margin: 8px 0 0 0; font-weight: 300;">Projeto 7 — Filtros Digitais</h3>
  <hr style="border: 1px solid white; margin: 16px auto; width: 60%; opacity: 0.5;">
  <p style="color: white; font-size: 1em; margin: 0;">Camada Física da Computação &nbsp;|&nbsp; Engenharia da Computação &nbsp;|&nbsp; Prof. Rodrigo Carareto</p>
</div>

---

## Sobre o Projeto

Um **filtro digital** é um sistema que processa sinais digitais com o objetivo de modificar ou extrair certas características do sinal. Ele atua como uma *peneira de frequências*: deixa passar apenas as frequências desejadas e bloqueia (ou amplifica) as demais.

Neste projeto, implementamos um **equalizador gráfico de áudio de 12 bandas**, similar ao que você encontra em aplicativos de música como Spotify ou VLC. O usuário pode configurar o ganho de cada banda de frequência entre **−10 dB e +10 dB**.

### Bandas de frequência

| Banda | Frequência Central | Região |
|-------|--------------------|--------|
| 1 | 32 Hz | Sub-grave |
| 2 | 64 Hz | Grave |
| 3 | 125 Hz | Grave médio |
| 4 | 250 Hz | Médio |
| 5 | 500 Hz | Médio |
| 6 | 1.000 Hz | Médio agudo |
| 7 | 2.000 Hz | Agudo |
| 8 | 4.000 Hz | Agudo |
| 9 | 8.000 Hz | Presença |
| 10 | 16.000 Hz | Brilho |
| 11 | 18.000 Hz | Ultra-agudo  |
| 12 | 20.000 Hz | Ultra-agudo  |

## Estrutura do Notebook

| # | Seção |
|---|-------|
| 1 | [Imports e Configurações](#1-imports) |
| 2 | [Carregamento do Áudio](#2-audio) |
| 3 | [Função do Filtro Peaking EQ](#3-peaking-eq) |
| 4 | [Interface com o Usuário (Sliders)](#4-sliders) |
| 5 | [Aplicação dos Filtros em Cascata](#5-cascata) |
| 6 | [Reprodução do Áudio](#6-reproducao) |
| 7 | [Diagrama de Bode do Filtro Completo](#7-bode) |

---

### 1. Imports e Configurações

In [ ]:
import numpy as np
from scipy.signal import iirpeak, freqz, TransferFunction, lfilter
import matplotlib.pyplot as plt
import sounddevice as sd
from scipy.io import wavfile
import ipywidgets as widgets
from IPython.display import display, Audio

---

### 2. Carregamento do Áudio

> O sinal de áudio eh uma sequência de amostras $X[k]$, captadas a uma taxa de amostragem $f_s = 44100$ Hz. Cada valor representa a  amplitude do sinal num instante discreto de tempo.

In [ ]:
duracao = 15 # segundos
fs = 44100 # freq de amostragem

print("Gravando em 3... 2... 1...")
audio_gravado = sd.rec(int(duracao * fs), samplerate=fs, channels=1, dtype='float64')
sd.wait()
print("Gravação concluída! Salvo como audio.wav")

# Salva o áudio gravado em disco
wavfile.write("audio.wav", fs, audio_gravado)

In [ ]:
# Carrega o arquivo de áudio (.wav)
# wavfile.read retorna a taxa de amostragem fs e o sinal X[k] como array numpy
fs, audio = wavfile.read("audio.wav")

# Se o áudio for estéreo (2 canais), converte para mono tirando a média dos canais.
# Isso simplifica o processamento — o filtro será aplicado em um único sinal 1D.
if audio.ndim == 2:
    audio = audio.mean(axis=1)

# Normaliza o sinal para o intervalo [-1.0, 1.0]
# Isso é importante para evitar overflow numérico ao aplicar os filtros
audio = audio.astype(np.float64)
# wavfile carrega o áudio como inteiro (int16 geralmente), 
# e a divisão pode dar problema sem converter antes para float.
audio = audio / np.max(np.abs(audio))

print(f"Taxa de amostragem: {fs} Hz")
print(f"Duração: {len(audio)/fs:.2f} segundos")
print(f"Total de amostras: {len(audio)}")

# Reproduz o áudio original para referência
print("\nReproduzindo áudio original...")
sd.play(audio, fs)
sd.wait()

---

### 3. Função do Filtro Peaking EQ

> O filtro ***Peaking EQ*** é o coração do equalizador. Para cada banda, ele gera uma função de transferência $G(Z)$ que amplifica ou atenua frequências próximas à uma frequência central $f0$, com intensidade definida pelo ganho em dB e largura de banda controlada pelo fator de qualidade $Q$. 

>O filtro Peaking EQ gera uma função de transferência $G(Z)$ do tipo **biquad (2ª ordem)**:

$$G(Z) = \frac{b_0 + b_1 Z^{-1} + b_2 Z^{-2}}{a_0 + a_1 Z^{-1} + a_2 Z^{-2}}$$

Os coeficientes $b$ (numerador) e $a$ (denominador) são calculados a partir da frequência central $f_0$, do ganho em dB e do fator de qualidade $Q$.

In [ ]:
def peaking_eq(f0, gain_db, Q, fs):
    """
    Calcula os coeficientes do filtro Peaking EQ.

    Parâmetros:
        f0       : frequência central da banda (Hz)
        gain_db  : ganho desejado em dB (positivo = amplifica, negativo = atenua)
        Q        : fator de qualidade — controla a largura da banda afetada
        fs       : taxa de amostragem (Hz)

    Retorna:
        b, a : coeficientes do numerador e denominador da função de transferência G(Z)
    """
    A = 10 ** (gain_db / 40)          # Amplitude linear correspondente ao ganho em dB
    omega = 2 * np.pi * f0 / fs       # Frequência angular normalizada
    alpha = np.sin(omega) / (2 * Q)   # Parâmetro de largura de banda

    # Coeficientes do numerador (b) e denominador (a)
    b0 =  1 + alpha * A
    b1 = -2 * np.cos(omega)
    b2 =  1 - alpha * A
    a0 =  1 + alpha / A
    a1 = -2 * np.cos(omega)
    a2 =  1 - alpha / A

    # Normaliza pelo a0 para que o denominador comece em 1
    b = np.array([b0, b1, b2]) / a0
    a = np.array([a0, a1, a2]) / a0

    return b, a


# Teste rápido: filtro centrado em 1000 Hz com ganho de +6 dB
b_teste, a_teste = peaking_eq(f0=1000, gain_db=6, Q=1.0, fs=fs)

w, h = freqz(b_teste, a_teste, fs=fs)
plt.figure(figsize=(8, 4))
plt.semilogx(w, 20 * np.log10(abs(h)))
plt.title("Peaking EQ — Teste: 1000 Hz, +6 dB")
plt.xlabel("Frequência [Hz]")
plt.ylabel("Ganho [dB]")
plt.ylim(-15, 15)
plt.grid(which='both', linestyle='--', linewidth=0.5)
plt.axvline(1000, color='red', linestyle=':', label='Frequência central')
plt.legend()
plt.tight_layout()
plt.show()

Plotando o gráfico do teste, vemos que:
- O pico esta em 1000Hz, que eh a freq central $f0$;
- O ganho nop pico eh $+6dB$, que eh oq configuramos;
- Fora da banda, o ganho eh $0dB$, então o filtro não afeta as outras frequências

---

### 4. Interface com o Usuário (Sliders)

> O equalizador possui 12 bandas de frequência. Para cada banda, o usuário configura o ganho entre $-10dB$ e $+10dB$. O fator de qualidade Q = 1.4 é um valor padrão comum em equalizadores gráficos de 12 bandas

In [ ]:
# Frequências centrais seguindo a ISO 266 e o enunciado do projeto (imagem do equalizador)
BANDAS = [32, 64, 125, 250, 500, 1000, 2000, 4000, 8000, 16000, 18000, 20000]
# As últimas duas bandas (18 kHz e 20 kHz) completam as 12 do enunciado 
# — o enunciado mostrou até 16 kHz na imagem mas pediu 12 bandas, então fechamos com 18k e 20k que são frequências audiáveis reais

# Q = 1/( sqrt(2) - 1/sqrt(2) ) ≈ 1.41 — valor teórico para equalizador gráfico de oitava
Q = 1.4

# Cria um slider para cada banda
sliders = []
for f in BANDAS:
    label = f"{f} Hz" if f < 1000 else f"{f//1000} kHz"
    slider = widgets.FloatSlider(
        value=0,
        min=-10,
        max=10,
        step=0.5,
        description=label,
        style={'description_width': '60px'},
        layout=widgets.Layout(width='400px')
    )
    sliders.append(slider)

# Botão para aplicar o equalizador
botao = widgets.Button(
    description="Aplicar Equalizador",
    button_style='primary',
    layout=widgets.Layout(width='200px', margin='15px 0px 0px 0px')
)

# Organiza os sliders em 2 colunas
coluna1 = widgets.VBox(sliders[:6])
coluna2 = widgets.VBox(sliders[6:])
painel = widgets.HBox([coluna1, coluna2])

# Exibe a interface
display(widgets.VBox([painel, botao]))

> ### Como usar o equalizador
> O equalizador possui **12 bandas de frequência**, cada uma controlada por um slider.
Cada slider representa uma frequência central e permite configurar o **ganho** daquela banda:
> - Mover o slider para **cima (direita)** → **amplifica** aquela faixa de frequência (até +10 dB)
> - Mover o slider para **baixo (esquerda)** → **atenua** aquela faixa de frequência (até -10 dB)
> - Slider no **centro (0 dB)** → aquela banda não é alterada
> Após configurar os sliders, clique em **"Aplicar Equalizador"** para processar o áudio.
>O programa irá então aplicar os 12 filtros Peaking EQ em cascata — ou seja, a saída $Y[k]$ de um filtro se torna a entrada $X[k]$ do próximo — e reproduzir o áudio filtrado logo em seguida.

---

### 5. Aplicação dos Filtros em Cascata

> Aqui, conectamos o botão "Aplicar Equalizador" à função que processa o áudio.
> Os 12 filtros Peaking EQ são aplicados em cascata: a saída $Y[k]$ de um filtro se torna a entrada $X[k]$ do próximo. Matematicamente, isso equivale a multiplicar as funções de transferência de cada banda:
>  $$G_{total}(Z) = G_1(Z) \cdot G_2(Z) \cdot \ldots \cdot G_{12}(Z)$$

> A função `lfilter(b, a, sinal)` aplica a equação de recorrência do filtro diretamente sobre o sinal discreto $X[k]$, produzindo $Y[k]$. 

---

### 6. Reprodução do Áudio

> Além de reproduzir os áudios, vamos plotar a FFT do sinal original e do filtrado. A FFT decompõe o sinal no domínio da frequência, mostrando quais frequências estão presentes e com qual amplitude. Isso permite verificar visualmente o efeito do equalizador.

In [ ]:
def aplicar_e_visualizar(b):
    # Lê o ganho de cada slider
    ganhos = [slider.value for slider in sliders]

    # Aplica os filtros em cascata
    audio_filtrado = audio.copy()
    for f0, gain_db in zip(BANDAS, ganhos):
        if gain_db != 0:
            b_coef, a_coef = peaking_eq(f0, gain_db, Q, fs)
            audio_filtrado = lfilter(b_coef, a_coef, audio_filtrado)

    # Normaliza
    audio_filtrado = audio_filtrado / np.max(np.abs(audio_filtrado))

    # FFT do sinal original e filtrado 
    N = len(audio)
    freqs = np.fft.rfftfreq(N, d=1/fs)         # Eixo de frequências
    fft_original = np.abs(np.fft.rfft(audio))   # Espectro do sinal original
    fft_filtrado = np.abs(np.fft.rfft(audio_filtrado))  # Espectro do filtrado

    # Plota as FFTs
    fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    axs[0].semilogx(freqs[1:], 20 * np.log10(fft_original[1:]), color='mediumvioletred')
    axs[0].set_title("Espectro do Áudio Original")
    axs[0].set_ylabel("Magnitude [dB]")
    axs[0].grid(which='both', linestyle='--', linewidth=0.5)

    axs[1].semilogx(freqs[1:], 20 * np.log10(fft_filtrado[1:]), color='palevioletred')
    axs[1].set_title("Espectro do Áudio Filtrado")
    axs[1].set_ylabel("Magnitude [dB]")
    axs[1].set_xlabel("Frequência [Hz]")
    axs[1].grid(which='both', linestyle='--', linewidth=0.5)

    plt.tight_layout()
    plt.show()

    # Reprodução dos áudios
    print("Reproduzindo áudio ORIGINAL...")
    sd.play(audio, fs)
    sd.wait()

    print("Reproduzindo áudio FILTRADO...")
    sd.play(audio_filtrado, fs)
    sd.wait()
    print("Concluído!")

# Reconecta o botão à nova função (com visualização)
botao.on_click(aplicar_e_visualizar)
print("Pronto! Configure os sliders e clique em 'Aplicar Equalizador'.")

---

### 7. Diagrama de Bode do Filtro Completo

> O diagrama de Bode do filtro completo mostra o efeito combinado de todos os 12 filtros ***Peaking EQ*** aplicados em cascata.
> Para obter a função de transferência equivalente, multiplicamos as funções de transferência individuais de cada banda:
> $$G_{total}(Z) = G_1(Z) \cdot G_2(Z) \cdot \ldots \cdot G_{12}(Z)$$

> Na prática, isso é feito calculando a resposta em frequência H(w) de cada filtro via freqz() e multiplicando os resultados complexos. O módulo de H_total(w) em dB nos dá o diagrama de Bode do equalizador completo.

In [ ]:
def plotar_bode(b):
    ganhos = [slider.value for slider in sliders]

    # Calcula a resposta em frequência total multiplicando as respostas individuais
    w, _ = freqz([1], [1], worN=8192, fs=fs)  # Define o eixo de frequências
    H_total = np.ones(len(w), dtype=complex)   # Começa com ganho 1 (neutro)

    for f0, gain_db in zip(BANDAS, ganhos):
        b_coef, a_coef = peaking_eq(f0, gain_db, Q, fs)
        _, H = freqz(b_coef, a_coef, worN=8192, fs=fs)
        H_total *= H  # Multiplica as funções de transferência em cascata

    # Plota o diagrama de Bode
    plt.figure(figsize=(10, 5))
    plt.semilogx(w, 20 * np.log10(np.abs(H_total)), color='hotpink', linewidth=2)
    plt.title("Diagrama de Bode — Filtro Completo (12 bandas)")
    plt.xlabel("Frequência [Hz]")
    plt.ylabel("Ganho [dB]")
    plt.ylim(-15, 15)
    plt.grid(which='both', linestyle='--', linewidth=0.5)

    # Marca as frequências centrais de cada banda
    for f0, gain_db in zip(BANDAS, ganhos):
        label = f"{f0} Hz" if f0 < 1000 else f"{f0//1000} kHz"
        plt.axvline(f0, color='gray', linestyle=':', linewidth=0.8)
        plt.annotate(label, xy=(f0, gain_db),
                     fontsize=7, ha='center', color='gray',
                     xytext=(0, 5), textcoords='offset points')

    plt.tight_layout()
    plt.show()

# Cria um botão separado para o Bode
botao_bode = widgets.Button(
    description="Plotar Diagrama de Bode",
    button_style='warning',
    layout=widgets.Layout(width='220px')
)
botao_bode._click_handlers.callbacks.clear()
botao_bode.on_click(plotar_bode)
display(botao_bode)
print("Configure os sliders na seção 4 e clique para ver o Bode do filtro completo.")